## Preprocessing edf format signals and hypnograms

1. Linking corresponding recording and hypnogram
2. Load data and convert sleep stages to numeric labels
3. Cut recordings before and after sleep
4. Convert to mne.Raw objects and save as .fif files

In [10]:
# imports
import os
import re
import mne
import numpy as np
from tqdm import tqdm

mne.set_log_level("WARNING")

In [ ]:
# define path to the data
# ADAPT THIS PATH TO WHERE YOU HAVE THE DATA
data_path = "../../data"

In [12]:
folder_path = "sleep-edfx/1.0.0"
sc = "sleep-cassette"
st = "sleep-telemetry"

sc_path = os.path.join(data_path, folder_path, sc)
st_path = os.path.join(data_path, folder_path, st)

In [13]:
# Function to extract subject ID as an integer from the edf file path
# e.g. for SC4001E0-PSG.edf extract 0 (participant 0)
def extract_subject_id(file_path):
    # extract subject number from file name
    pattern = r'SC4(\d{2})\d'
    match = re.search(pattern, file_path)
    if match:
        return int(match.group(1)) # return subject ID as integer (0 would be intire matched string)
    return None


# get lists of psg file paths and corresponding hypnogram file paths
def get_data_paths(directory):
    all_files = os.listdir(directory)

    psg_files = [f for f in all_files if f.endswith('PSG.edf')]
    hypnogram_files = [f for f in all_files if f.endswith('Hypnogram.edf')]
    
    sorted_psg_paths = []
    sorted_hypnogram_paths = []

    # pair psg and hypnogram files based on participant and night name -> there is always a pair of PSG and Hypnogram files
    # e.g. for SC4001E0-PSG.edf extract identifier SC4001
    for psg_file in psg_files:
        base_name = psg_file.split('-')[0]
        identifier = base_name[:-2]
        subject_id = extract_subject_id(psg_file)
        for hypnogram_file in hypnogram_files:
            if hypnogram_file.startswith(identifier):
                sorted_psg_paths.append(os.path.join(directory, psg_file))
                sorted_hypnogram_paths.append(os.path.join(directory, hypnogram_file))

    # returns 2 lists of paths (1 for PSG 1 for Hypnograms) sorted in the same order
    return sorted_psg_paths, sorted_hypnogram_paths


In [14]:
sc_psg_paths, sc_hypnogram_paths = get_data_paths(sc_path)
st_psg_paths, st_hypnogram_paths = get_data_paths(st_path)

In [15]:
# convert sleep stage (string) to numeric stage
def map_sleep_stage_to_label(stage):
    stage_mapping = {
    'Sleep stage W': 0,
    'Sleep stage R': 1,
    'Sleep stage 1': 2,
    'Sleep stage 2': 3,
    'Sleep stage 3': 4,
    'Sleep stage 4': 5
    }
    return stage_mapping.get(stage, -1) # -1 is Sleep Stage ? or Movement time

# load data from edf files
def load_data(psg_path, hypnogram_path):
    
    raw = mne.io.read_raw_edf(psg_path, preload=False, verbose='ERROR')

    # # choose channels I want to use -> keeping all for now
    # print(raw.info['ch_names'])
    # channels = ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'EMG submental']
    # raw.pick_channels(channels)

    psg_data = raw.get_data()

    hypnogram = mne.read_annotations(hypnogram_path)

    n_samples = len(raw.times)
    sfreq = raw.info['sfreq'] # is 100 Hz

    ch_names = raw.info['ch_names']
    ch_types = raw.get_channel_types()

    sleep_stage_labels = np.zeros(n_samples, dtype=np.int32)

    # create list containing sleep stage for every time point
    for row in hypnogram:
        onset_sample = int(row['onset'] * sfreq)
        duration_sample = int(row['duration'] * sfreq)
        stage = row['description']

        stage_label = map_sleep_stage_to_label(stage)

        sleep_stage_labels[onset_sample:onset_sample + duration_sample] = stage_label

    return (psg_data, sleep_stage_labels, sfreq, ch_names, ch_types)

In [16]:
# removing wake stages before and after sleep
def remove_wake(signals, labels, wake_stage_index=0, sampling_rate=100):

    # find indices of all non-wake stages
    non_wake_indices = np.where(labels != wake_stage_index)[0]

    if len(non_wake_indices) == 0:
        raise ValueError("No non-wake stages found in the data.")
    
    # get first and last non-wake stage indices
    first_non_wake_index = non_wake_indices[0]
    last_non_wake_index = non_wake_indices[-1]

    # calculate start and end of period for trimming
    start_index = max(0, first_non_wake_index)
    end_index = min(signals.shape[1], last_non_wake_index)

    # trim signals and labels
    trimmed_signals = signals[:, start_index:end_index]
    trimmed_labels = labels[start_index:end_index]

    return trimmed_signals, trimmed_labels

In [17]:
def hypnogram_to_annotations(labels, sfreq):
    annotations = []
    current_label = labels[0]
    start = 0

    for i in range(1, len(labels)):
        if labels[i] != current_label:
            duration = (i - start) / sfreq
            onset = start / sfreq
            annotations.append((onset, duration, str(current_label)))
            current_label = labels[i]
            start = i

    # last segment
    duration = (len(labels) - start) / sfreq
    onset = start / sfreq
    annotations.append((onset, duration, str(current_label)))

    onsets, durations, descriptions = zip(*annotations)
    return mne.Annotations(onsets, durations, descriptions)

In [ ]:
for psg_path, hypnogram_path in tqdm(zip(sc_psg_paths, sc_hypnogram_paths), desc="Processing sleep-cassette data", total=len(sc_psg_paths)):
    signal, label, sfreq, ch_names, ch_types = load_data(psg_path, hypnogram_path)
    trimmed_sig, trimmed_labels = remove_wake(signal, label)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw_new = mne.io.RawArray(trimmed_sig, info)
    annot = hypnogram_to_annotations(trimmed_labels, sfreq)
    raw_new.set_annotations(annot)
    out_dir = sc_path + "/preprocessed/"
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.basename(psg_path).replace("PSG.edf", "raw.fif")
    out_path = os.path.join(out_dir, fname)
    raw_new.save(out_path, overwrite=True)

for psg_path, hypnogram_path in tqdm(zip(st_psg_paths, st_hypnogram_paths), desc="Processing sleep-telemetry data", total=len(st_psg_paths)):
    signal, label, sfreq, ch_names, ch_types = load_data(psg_path, hypnogram_path)
    trimmed_sig, trimmed_labels = remove_wake(signal, label)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw_new = mne.io.RawArray(trimmed_sig, info)
    annot = hypnogram_to_annotations(trimmed_labels, sfreq)
    raw_new.set_annotations(annot)
    out_dir = st_path + "/preprocessed/"
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.basename(psg_path).replace("PSG.edf", "raw.fif")
    out_path = os.path.join(out_dir, fname)
    raw_new.save(out_path, overwrite=True)


Processing sleep-cassette data:   9%|▉         | 14/153 [00:24<03:49,  1.65s/it]